# LA Daily AQI Pipeline (2016–2025)

Building a city-wide daily AQI series for Los Angeles County from EPA AirData exports covering ozone and PM2.5.

**Note on re-execution.** This notebook has already been run end-to-end. The output files (`raw_renamed/`, `la_only/`, `la_daily_aqi_2016_2025.csv`) exist on disk. The file-writing cells (rename, filter, save) are idempotent — running them again just overwrites with the same content — but you don't need to re-execute the notebook to inspect results; outputs are preserved in the cells below.

## Setup

Paths to the two raw EPA folders, plus the intermediate and final output locations.

In [1]:
import os, glob, shutil
from pathlib import Path
import numpy as np
import pandas as pd

# ROOT is the repo directory. Anchored to the working directory so this runs for
# anyone who clones the repo, whether Jupyter was launched from the repo root or
# from notebooks/.
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'data' / 'raw' / 'epa_ozone').exists(), (
    f'Expected data/raw/epa_ozone under {ROOT}. '
    'Launch Jupyter from the repo root or notebooks/, or set ROOT manually.'
)

DATA        = ROOT / 'data'
RAW         = DATA / 'raw'
INTERIM     = DATA / 'interim'
PROCESSED   = DATA / 'processed'
CORRECTIONS = DATA / 'corrections'

OZONE_DIR   = RAW / 'epa_ozone'
POLL_DIR    = RAW / 'epa_pollutant'
RAW_RENAMED = INTERIM / 'raw_renamed'
LA_ONLY     = INTERIM / 'la_only'
API_DIR     = INTERIM / 'api_pulls'
OUT_CSV     = PROCESSED / 'la_daily_aqi_2016_2025.csv'

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

ozone_files = sorted(glob.glob(str(OZONE_DIR / '*.csv')))
poll_files  = sorted(glob.glob(str(POLL_DIR  / '*.csv')))
print(f'{len(ozone_files)} ozone files, {len(poll_files)} pollutant files')

13 ozone files, 13 pollutant files


## Inspecting the raw files

Before touching anything, we check what we have: how many files each folder holds, which calendar year each file covers, whether all files in a folder share the same columns, and — critically for the pollutant folder — whether each file holds a single pollutant or a mix. Doing this up front prevents wrong assumptions from propagating downstream.

In [2]:
def summarize(files, folder_label):
    rows, col_sets = [], {}
    for f in files:
        df = pd.read_csv(f, low_memory=False)
        col_sets[os.path.basename(f)] = tuple(df.columns)
        dates = pd.to_datetime(df['Date'], errors='coerce')
        years = sorted(dates.dt.year.dropna().unique().astype(int).tolist())
        pollutants = sorted(df['AQS Parameter Description'].dropna().unique().tolist())
        rows.append({
            'folder': folder_label,
            'filename': os.path.basename(f),
            'years': years,
            'min_date': dates.min(),
            'max_date': dates.max(),
            'pollutants': pollutants,
            'n_pollutants': len(pollutants),
            'rows': len(df),
        })
    print(f'--- {folder_label}: {len(files)} files, {len(set(col_sets.values()))} distinct column layouts')
    return pd.DataFrame(rows)

oz_summary   = summarize(ozone_files, 'ozone')
poll_summary = summarize(poll_files,  'pollutant')

--- ozone: 13 files, 1 distinct column layouts


--- pollutant: 13 files, 1 distinct column layouts


In [3]:
print('OZONE SUMMARY')
print(oz_summary.to_string(index=False))
print()
print('POLLUTANT SUMMARY')
print(poll_summary.to_string(index=False))

OZONE SUMMARY
folder                   filename  years   min_date   max_date pollutants  n_pollutants  rows
 ozone ad_viz_plotval_data-10.csv [2023] 2023-01-01 2023-12-30    [Ozone]             1  4932
 ozone ad_viz_plotval_data-11.csv [2024] 2024-01-01 2024-12-30    [Ozone]             1  4919
 ozone ad_viz_plotval_data-12.csv [2025] 2025-01-01 2025-12-31    [Ozone]             1  5067
 ozone ad_viz_plotval_data-13.csv [2026] 2026-01-01 2026-06-24    [Ozone]             1  2371
 ozone  ad_viz_plotval_data-2.csv [2015] 2015-01-01 2015-12-30    [Ozone]             1  5756
 ozone  ad_viz_plotval_data-3.csv [2016] 2016-01-01 2016-12-30    [Ozone]             1  5968
 ozone  ad_viz_plotval_data-4.csv [2017] 2017-01-01 2017-12-30    [Ozone]             1  5616
 ozone  ad_viz_plotval_data-5.csv [2018] 2018-01-01 2018-12-30    [Ozone]             1  5630
 ozone  ad_viz_plotval_data-6.csv [2019] 2019-01-01 2019-12-30    [Ozone]             1  5581
 ozone  ad_viz_plotval_data-7.csv [2020] 2020-

**What this told us.** Both folders have 13 files with one file per calendar year covering 2014–2026 (2026 is partial through June). Columns are consistent within each folder. Every ozone file is single-parameter Ozone. Every pollutant file contains *two* PM2.5 parameter descriptions (`PM2.5 - Local Conditions` and `Acceptable PM2.5 AQI & Speciation Mass`) — both are PM2.5, just different EPA parameter codes, and we'll collapse them to a single label later. So the pollutant folder is effectively all PM2.5 — no SO₂, NO₂, CO, or PM10 mixed in.

### Checking for year duplicates and gaps

In [4]:
def year_check(summary, label):
    all_years = [y for ys in summary['years'] for y in ys]
    all_years_sorted = sorted(all_years)
    dupes = [y for y in set(all_years) if all_years.count(y) > 1]
    full_range = list(range(min(all_years_sorted), max(all_years_sorted)+1))
    missing = [y for y in full_range if y not in all_years_sorted]
    print(f'[{label}] year coverage: {min(all_years_sorted)}–{max(all_years_sorted)} '
          f'({len(set(all_years_sorted))} unique yrs)')
    print(f'   duplicate years : {dupes if dupes else "none"}')
    print(f'   missing years   : {missing if missing else "none"}')

year_check(oz_summary,   'ozone')
year_check(poll_summary, 'pollutant')

[ozone] year coverage: 2014–2026 (13 unique yrs)
   duplicate years : none
   missing years   : none
[pollutant] year coverage: 2014–2026 (13 unique yrs)
   duplicate years : none
   missing years   : none


No duplicate years, no gaps in either folder — clean 13-year sequence.

## Renaming files by year

Filenames like `ad_viz_plotval_data-8.csv` carry no information. We copy them into a new `raw_renamed/` folder using a clean `ozone_YYYY.csv` / `pm25_YYYY.csv` convention so downstream code can address years directly. We also scope the working dataset to the ten complete years **2016–2025**, dropping 2014, 2015, and partial 2026. Originals stay untouched.

In [5]:
KEEP_YEARS = set(range(2016, 2026))
os.makedirs(RAW_RENAMED, exist_ok=True)

def rename_files(summary, prefix, src_dir):
    plan = []
    for _, row in summary.iterrows():
        if len(row['years']) != 1:
            plan.append((row['filename'], None, f"SKIP: multiple years {row['years']}"))
            continue
        yr = row['years'][0]
        if yr not in KEEP_YEARS:
            plan.append((row['filename'], None, f"SKIP: {yr} outside 2016–2025 window"))
            continue
        new_name = f'{prefix}_{yr}.csv'
        shutil.copy2(os.path.join(src_dir, row['filename']),
                     os.path.join(RAW_RENAMED, new_name))
        plan.append((row['filename'], new_name, 'OK'))
    return plan

oz_plan   = rename_files(oz_summary,   'ozone', OZONE_DIR)
poll_plan = rename_files(poll_summary, 'pm25',  POLL_DIR)

for label, plan in [('OZONE', oz_plan), ('POLLUTANT', poll_plan)]:
    print(f'=== {label} rename plan ===')
    for src, dst, note in plan:
        print(f'  {src:32s} -> {dst if dst else "(skip)":18s}  {note}')
    print()
print(f'Files now in raw_renamed/: {len(os.listdir(RAW_RENAMED))}')

=== OZONE rename plan ===
  ad_viz_plotval_data-10.csv       -> ozone_2023.csv      OK
  ad_viz_plotval_data-11.csv       -> ozone_2024.csv      OK
  ad_viz_plotval_data-12.csv       -> ozone_2025.csv      OK
  ad_viz_plotval_data-13.csv       -> (skip)              SKIP: 2026 outside 2016–2025 window
  ad_viz_plotval_data-2.csv        -> (skip)              SKIP: 2015 outside 2016–2025 window
  ad_viz_plotval_data-3.csv        -> ozone_2016.csv      OK
  ad_viz_plotval_data-4.csv        -> ozone_2017.csv      OK
  ad_viz_plotval_data-5.csv        -> ozone_2018.csv      OK
  ad_viz_plotval_data-6.csv        -> ozone_2019.csv      OK
  ad_viz_plotval_data-7.csv        -> ozone_2020.csv      OK
  ad_viz_plotval_data-8.csv        -> ozone_2021.csv      OK
  ad_viz_plotval_data-9.csv        -> ozone_2022.csv      OK
  ad_viz_plotval_data.csv          -> (skip)              SKIP: 2014 outside 2016–2025 window

=== POLLUTANT rename plan ===
  ad_viz_plotval_data-10.csv       -> pm25_2023.csv

## Verifying county spellings

The raw exports are pulled from CBSA-based EPA queries and can pick up surrounding counties. Before filtering, we list every unique `County` value across all files so we catch typos, trailing-space variants, or unexpected counties. "Los Angeles" needs to actually mean Los Angeles.

In [6]:
renamed_files = sorted(glob.glob(os.path.join(RAW_RENAMED, '*.csv')))

county_counts = {}
for f in renamed_files:
    df = pd.read_csv(f, low_memory=False, usecols=['County'])
    for c, n in df['County'].value_counts(dropna=False).items():
        county_counts[c] = county_counts.get(c, 0) + int(n)

county_df = (pd.DataFrame({'County': list(county_counts.keys()),
                           'rows_across_all_files': list(county_counts.values())})
               .sort_values('rows_across_all_files', ascending=False))
print(county_df.to_string(index=False))

     County  rows_across_all_files
Los Angeles                  89574
     Orange                  18134
  Jefferson                    827
      Clark                    363
      Floyd                    361
     Oldham                    242
    Bullitt                    233


**What this told us.** "Los Angeles" is spelled consistently (no typos, no trailing spaces). Orange County appears as expected (same CBSA export). And unexpectedly, one file contained rows from Jefferson KY, Clark IN, Floyd IN, Oldham KY, and Bullitt KY — a Louisville-area file that was misplaced in the LA ozone folder. We handle that below.

## Removing other counties

Straight filter on `County == 'Los Angeles'`. Filtered versions are written to `la_only/` so the pre-filter files stay intact. We log before/after row counts per file to see exactly how much Orange County (and misplaced-file) data drops out.

In [7]:
os.makedirs(LA_ONLY, exist_ok=True)

rows_log = []
for f in renamed_files:
    name = os.path.basename(f)
    df = pd.read_csv(f, low_memory=False)
    before = len(df)
    la = df[df['County'] == 'Los Angeles'].copy()
    after = len(la)
    la.to_csv(os.path.join(LA_ONLY, name), index=False)
    rows_log.append({'file': name, 'rows_before': before,
                     'rows_after_LA': after, 'rows_dropped': before - after,
                     'pct_kept': round(100 * after / before, 1) if before else 0.0})

rows_log_df = pd.DataFrame(rows_log)
print(rows_log_df.to_string(index=False))
print(f"\nTotals: before={rows_log_df['rows_before'].sum():,}, "
      f"after={rows_log_df['rows_after_LA'].sum():,}, "
      f"dropped={rows_log_df['rows_dropped'].sum():,}")

          file  rows_before  rows_after_LA  rows_dropped  pct_kept
ozone_2016.csv         5968           4571          1397      76.6
ozone_2017.csv         5616           4390          1226      78.2
ozone_2018.csv         5630           4562          1068      81.0
ozone_2019.csv         5581           4511          1070      80.8
ozone_2020.csv         5886           4823          1063      81.9
ozone_2021.csv         2026              0          2026       0.0
ozone_2022.csv         5418           4497           921      83.0
ozone_2023.csv         4932           4239           693      85.9
ozone_2024.csv         4919           4205           714      85.5
ozone_2025.csv         5067           4267           800      84.2
 pm25_2016.csv         5600           4726           874      84.4
 pm25_2017.csv         5648           4785           863      84.7
 pm25_2018.csv         5688           4768           920      83.8
 pm25_2019.csv         5801           4918           883      

About 18% of rows drop out — mostly Orange County. **`ozone_2021.csv` drops to zero LA rows** because the entire file was Kentucky/Indiana data. That's fixed next.

## Correcting the misplaced 2021 ozone file

The original `ad_viz_plotval_data-8.csv` in the ozone folder turned out to be a Louisville-metro download — 100% Kentucky and Indiana monitors, zero California rows. We re-downloaded the correct 2021 LA ozone file directly from EPA AirData (4,809 rows, all Los Angeles County, full year) and committed it to the repo at `raw_corrections/ozone_2021_la.csv`. The cell below overwrites the misfiled version in `raw_renamed/` and its downstream copy in `la_only/`, so anyone who clones the repo gets the corrected data automatically.

In [8]:
# The corrected 2021 LA ozone file lives in the repo at raw_corrections/ozone_2021_la.csv.
# Anyone who clones the repo gets it. This cell overwrites the misfiled version in raw_renamed/
# (and its downstream copy in la_only/) with the good one.
CORRECTION_SRC = CORRECTIONS / 'ozone_2021_la.csv'

if not CORRECTION_SRC.exists():
    raise FileNotFoundError(f'Missing correction file: {CORRECTION_SRC}')

shutil.copy2(CORRECTION_SRC, RAW_RENAMED / 'ozone_2021.csv')
df = pd.read_csv(RAW_RENAMED / 'ozone_2021.csv', low_memory=False)
la = df[df['County'] == 'Los Angeles'].copy()
la.to_csv(LA_ONLY / 'ozone_2021.csv', index=False)

print(f'Corrected 2021 ozone. Rows: {len(df):,} total, {len(la):,} LA')
print(f'States in file: {sorted(df["State"].unique().tolist())}')

Corrected 2021 ozone. Rows: 4,809 total, 4,809 LA
States in file: ['California']


## Merging ozone and PM2.5 into one long table

We stack all twenty LA-only files into a single tidy table with one row per monitor-day: `date, site, site_id, pollutant_type, aqi, percent_complete, source_file`. The two PM2.5 parameter descriptions are collapsed into a single `PM2.5` label so downstream aggregation treats them as one pollutant.

In [9]:
def normalize_pollutant(desc):
    if desc == 'Ozone': return 'Ozone'
    if desc in ('PM2.5 - Local Conditions', 'Acceptable PM2.5 AQI & Speciation Mass'):
        return 'PM2.5'
    return desc

frames = []
for f in sorted(glob.glob(os.path.join(LA_ONLY, '*.csv'))):
    df = pd.read_csv(f, low_memory=False)
    if len(df) == 0:
        continue
    frames.append(pd.DataFrame({
        'date': pd.to_datetime(df['Date'], errors='coerce'),
        'site': df['Local Site Name'],
        'site_id': df['Site ID'],
        'pollutant_type': df['AQS Parameter Description'].map(normalize_pollutant),
        'aqi': pd.to_numeric(df['Daily AQI Value'], errors='coerce'),
        'percent_complete': pd.to_numeric(df['Percent Complete'], errors='coerce'),
        'source_file': os.path.basename(f),
    }))

long = pd.concat(frames, ignore_index=True)
print(f'Merged rows: {len(long):,}')
print()
print('Pollutant breakdown:')
print(long['pollutant_type'].value_counts().to_string())
print()
print('Rows by year and pollutant:')
print(long.assign(year=long['date'].dt.year)
         .groupby(['year','pollutant_type']).size().unstack(fill_value=0).to_string())

Merged rows: 94,383

Pollutant breakdown:
pollutant_type
PM2.5    49509
Ozone    44874

Rows by year and pollutant:
pollutant_type  Ozone  PM2.5
year                        
2016             4571   4726
2017             4390   4785
2018             4562   4768
2019             4511   4918
2020             4823   5366
2021             4809   5508
2022             4497   5070
2023             4239   4824
2024             4205   4787
2025             4267   4757


## Data quality checks

Sanity pass on the merged table: any missing values? Any AQI values outside EPA's valid 0–500 range? And how completeness is distributed — this determines whether the completeness threshold we're about to apply will be a heavy filter or a formality.

In [10]:
print('Missing values:')
print(long.isna().sum().to_string())
print()
print(f'AQI < 0   : {(long["aqi"] < 0).sum()}')
print(f'AQI > 500 : {(long["aqi"] > 500).sum()}')
print()
print('Percent Complete distribution:')
print(long['percent_complete'].describe().to_string())
print()
print('Rows by Percent Complete bin:')
bins = [-0.01, 25, 50, 75, 90, 99, 100.01]
labels = ['<25', '25–50', '50–75', '75–90', '90–99', '>=99']
print(pd.cut(long['percent_complete'], bins=bins, labels=labels).value_counts().sort_index().to_string())

Missing values:
date                0
site                0
site_id             0
pollutant_type      0
aqi                 0
percent_complete    0
source_file         0

AQI < 0   : 0
AQI > 500 : 0

Percent Complete distribution:
count    94383.000000
mean        99.732303
std          2.350094
min          6.000000
25%        100.000000
50%        100.000000
75%        100.000000
max        100.000000

Rows by Percent Complete bin:
percent_complete
<25          2
25–50        3
50–75       27
75–90     1233
90–99       82
>=99     93036


No missing values, no out-of-range AQIs, and roughly 99% of monitor-days report 100% completeness — the data is very clean.

## Applying the completeness threshold

We drop any monitor-day where `Percent Complete < 75`. This is the standard EPA-adjacent threshold for treating a day's reading as representative. Given the completeness distribution above, this only removes 19 rows out of 94,383 — effectively a formality here, not a heavy filter.

In [11]:
THRESHOLD = 75.0
before = len(long)
long_ok = long[(long['percent_complete'] >= THRESHOLD) & long['aqi'].notna()].copy()
after = len(long_ok)
print(f'Rows before threshold : {before:,}')
print(f'Rows after  threshold : {after:,}')
print(f'Dropped               : {before-after:,}  ({100*(before-after)/before:.2f}%)')

Rows before threshold : 94,383
Rows after  threshold : 94,364
Dropped               : 19  (0.02%)


## Computing the city-wide daily AQI

For each date, the city-wide daily AQI is the **maximum** AQI across every monitor and every pollutant. This mirrors how EPA officially reports "today's AQI" for a metro area — the worst pollutant on the worst monitor sets the number. We also record which pollutant and which site drove that maximum, so downstream analysis can attribute bad days to ozone events vs. PM2.5 smoke events.

In [12]:
daily = (long_ok.groupby('date', as_index=False)
               .agg(daily_aqi=('aqi', 'max'),
                    n_monitor_readings=('aqi', 'count'),
                    n_sites=('site_id', 'nunique'),
                    pollutants_used=('pollutant_type', lambda s: ','.join(sorted(s.unique())))))

idx_max = long_ok.groupby('date')['aqi'].idxmax()
dom = long_ok.loc[idx_max, ['date','pollutant_type','site']].rename(
    columns={'pollutant_type':'dominant_pollutant', 'site':'dominant_site'})
daily = daily.merge(dom, on='date', how='left').sort_values('date').reset_index(drop=True)

print(f'Daily AQI rows: {len(daily):,}')
print()
print(daily.head().to_string(index=False))
print('...')
print(daily.tail().to_string(index=False))

Daily AQI rows: 3,653

      date  daily_aqi  n_monitor_readings  n_sites pollutants_used dominant_pollutant                  dominant_site
2016-01-01        101                  31       17     Ozone,PM2.5              PM2.5 Long Beach-Route 710 Near Road
2016-01-02         83                  23       16     Ozone,PM2.5              PM2.5                         Reseda
2016-01-03        100                  23       16     Ozone,PM2.5              PM2.5 Long Beach-Route 710 Near Road
2016-01-04         68                  27       16     Ozone,PM2.5              PM2.5                         Reseda
2016-01-05         58                  22       16     Ozone,PM2.5              PM2.5                         Reseda
...
      date  daily_aqi  n_monitor_readings  n_sites pollutants_used dominant_pollutant                  dominant_site
2025-12-27         66                  30       13     Ozone,PM2.5              PM2.5                        Compton
2025-12-28         72                

## Coverage and distribution checks

Two things to confirm before saving: (1) every single day in the 2016-01-01 to 2025-12-31 range has an AQI value, and (2) the distribution looks like real LA data — mostly moderate, a long tail into unhealthy territory during summer smoke and ozone events, no impossible outliers.

In [13]:
full_range = pd.date_range('2016-01-01', '2025-12-31', freq='D')
missing = full_range.difference(pd.DatetimeIndex(daily['date']))
print(f'Expected days: {len(full_range):,}')
print(f'Days with AQI value: {len(daily):,}')
print(f'Missing days: {len(missing):,}')

Expected days: 3,653
Days with AQI value: 3,653
Missing days: 0


In [14]:
print('Daily AQI distribution:')
print(daily['daily_aqi'].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).to_string())
print()
cats = pd.cut(daily['daily_aqi'],
              bins=[-1, 50, 100, 150, 200, 300, 500],
              labels=['Good (0–50)', 'Moderate (51–100)', 'USG (101–150)',
                     'Unhealthy (151–200)', 'Very Unhealthy (201–300)', 'Hazardous (301–500)'])
print('AQI category counts (EPA breakpoints):')
print(cats.value_counts().sort_index().to_string())
print()
print('Top 10 highest AQI days (candidate smoke / ozone events):')
print(daily.nlargest(10, 'daily_aqi').to_string(index=False))

Daily AQI distribution:
count    3653.000000
mean       88.474678
std        38.206173
min        31.000000
50%        75.000000
75%       108.000000
90%       150.000000
95%       169.000000
99%       202.480000
max       250.000000

AQI category counts (EPA breakpoints):
daily_aqi
Good (0–50)                  255
Moderate (51–100)           2341
USG (101–150)                703
Unhealthy (151–200)          309
Very Unhealthy (201–300)      45
Hazardous (301–500)            0

Top 10 highest AQI days (candidate smoke / ozone events):
      date  daily_aqi  n_monitor_readings  n_sites pollutants_used dominant_pollutant                 dominant_site
2020-07-05        250                  33       18     Ozone,PM2.5              PM2.5 Los Angeles-North Main Street
2020-09-05        235                  27       16     Ozone,PM2.5              Ozone                      Glendora
2024-07-05        230                  31       13     Ozone,PM2.5              PM2.5                      Glen

In [15]:
annual = (daily.assign(year=daily['date'].dt.year)
               .groupby('year')
               .agg(days=('daily_aqi','size'),
                    mean_aqi=('daily_aqi','mean'),
                    median_aqi=('daily_aqi','median'),
                    max_aqi=('daily_aqi','max'),
                    days_over_100=('daily_aqi', lambda s: (s>100).sum()),
                    days_over_150=('daily_aqi', lambda s: (s>150).sum())))
annual['mean_aqi']   = annual['mean_aqi'].round(1)
annual['median_aqi'] = annual['median_aqi'].round(1)
print('Annual summary:')
print(annual.to_string())

Annual summary:
      days  mean_aqi  median_aqi  max_aqi  days_over_100  days_over_150
year                                                                   
2016   366      87.6        76.0      226            104             23
2017   365      93.1        78.0      224            118             46
2018   365      87.2        77.0      201            108             19
2019   365      83.3        71.0      201             83             29
2020   366      98.9        84.5      250            136             60
2021   365      88.0        77.0      195             97             26
2022   365      84.9        73.0      209             94             29
2023   365      82.8        67.0      210             87             35
2024   366      94.1        79.5      230            133             55
2025   365      84.8        73.0      201             97             32


All 3,653 expected days are present. The distribution is centered around a median of 75 (Moderate), with a long tail — the worst 10 days are all summer smoke or ozone events, dominated by the 2020 fire season. 2020 and 2024 are the two worst years overall (highest means, most days > 150). This matches known LA air-quality patterns.

## Saving the final dataset

In [16]:
daily.to_csv(OUT_CSV, index=False)
print(f'Saved: {OUT_CSV}')
print(f'Rows : {len(daily):,}   Size: {os.path.getsize(OUT_CSV)/1024:.1f} KB')

Saved: /Users/bealuzenebe/Desktop/Research/altREU---Team-Bullets/data/processed/la_daily_aqi_2016_2025.csv
Rows : 3,653   Size: 204.3 KB


## Fetching daily data from the EPA AQS API

To go beyond ozone and PM2.5, we hit the EPA AQS API directly and pull daily data for five pollutants (Ozone, PM2.5, CO, SO2, NO2) across 2016–2025 — 50 requests total. Credentials come from environment variables `AQS_EMAIL` and `AQS_KEY` (with an interactive `getpass` fallback), so the key never gets baked into the notebook file. We sleep 6 seconds between requests to stay under EPA's rate limits, so the full pull takes about 5 minutes. Each response is written to `api_pulls/{pollutant}_{year}_api.csv`, and before saving we assert every row is actually California / Los Angeles County — the same kind of check that would have caught the misfiled 2021 Louisville file. Nothing in `raw_renamed/`, `la_only/`, or `raw_corrections/` is touched.

In [17]:
import time, requests, getpass

# --- Load credentials from .env (kept out of git via .gitignore), env vars, or getpass ---
def _load_dotenv(path):
    """Populate os.environ from a simple KEY=VALUE .env file, if present."""
    if not path.exists():
        return
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        k, v = line.split('=', 1)
        os.environ.setdefault(k.strip(), v.strip())

_load_dotenv(ROOT / '.env')

AQS_EMAIL = os.environ.get('AQS_EMAIL')
AQS_KEY   = os.environ.get('AQS_KEY')
if not (AQS_EMAIL and AQS_KEY):
    AQS_EMAIL = AQS_EMAIL or input('AQS API email: ').strip()
    AQS_KEY   = AQS_KEY   or getpass.getpass('AQS API key (hidden): ').strip()

API_URL       = 'https://aqs.epa.gov/data/api/dailyData/byCounty'
STATE, COUNTY = '06', '037'          # California, Los Angeles
YEARS         = range(2016, 2026)
POLLUTANTS    = {
    'ozone': '44201',
    'pm25':  '88101',
    'co':    '42101',
    'so2':   '42401',
    'no2':   '42602',
}
DELAY_SEC = 6.0                       # >= 5s to respect EPA rate limits

API_DIR = API_DIR
API_DIR.mkdir(exist_ok=True)

results = []
n_total = len(POLLUTANTS) * len(list(YEARS))
print(f'Fetching {n_total} requests, {DELAY_SEC}s apart '
      f'(~{n_total*DELAY_SEC/60:.1f} min)\n')

for pname, pcode in POLLUTANTS.items():
    for year in YEARS:
        bdate, edate = f'{year}0101', f'{year}1231'
        params = {'email': AQS_EMAIL, 'key': AQS_KEY,
                  'param': pcode, 'bdate': bdate, 'edate': edate,
                  'state': STATE, 'county': COUNTY}
        try:
            r = requests.get(API_URL, params=params, timeout=60)
            status = r.status_code
            n_rows = 0
            note = ''
            if status != 200:
                note = f'HTTP {status}: {r.text[:120]}'
            else:
                payload = r.json()
                header_status = payload.get('Header', [{}])[0].get('status', '')
                data = payload.get('Data', []) or []
                n_rows = len(data)
                if n_rows == 0:
                    note = f'empty (API says: {header_status})'
                else:
                    df_api = pd.DataFrame(data)
                    assert 'state'  in df_api.columns, f'{pname} {year}: no "state" column in response'
                    assert 'county' in df_api.columns, f'{pname} {year}: no "county" column in response'
                    bad_state  = (df_api['state']  != 'California').sum()
                    bad_county = (df_api['county'] != 'Los Angeles').sum()
                    assert bad_state == 0 and bad_county == 0, (
                        f'{pname} {year}: {bad_state} non-CA rows / '
                        f'{bad_county} non-LA rows — refusing to save.'
                    )
                    out = API_DIR / f'{pname}_{year}_api.csv'
                    df_api.to_csv(out, index=False)
                    note = f'saved {out.name}'
            print(f'  {pname:6s} {year}  HTTP {status}  rows={n_rows:<6d}  {note}')
            results.append({'pollutant': pname, 'year': year,
                            'http': status, 'rows': n_rows, 'note': note})
        except Exception as e:
            print(f'  {pname:6s} {year}  ERROR: {type(e).__name__}: {e}')
            results.append({'pollutant': pname, 'year': year,
                            'http': None, 'rows': 0, 'note': f'ERROR: {e}'})
        time.sleep(DELAY_SEC)

print(f'\nDone. {len(results)} requests completed.')

Fetching 50 requests, 6.0s apart (~5.0 min)



  ozone  2016  HTTP 200  rows=18911   saved ozone_2016_api.csv


  ozone  2017  HTTP 200  rows=18298   saved ozone_2017_api.csv


  ozone  2018  HTTP 200  rows=18886   saved ozone_2018_api.csv


  ozone  2019  HTTP 200  rows=18756   saved ozone_2019_api.csv


...

  no2    2022  HTTP 200  rows=10718   saved no2_2022_api.csv


  no2    2023  HTTP 200  rows=10206   saved no2_2023_api.csv


  no2    2024  HTTP 200  rows=10100   saved no2_2024_api.csv


  no2    2025  HTTP 200  rows=10130   saved no2_2025_api.csv



Done. 50 requests completed.


In [18]:
res_df = pd.DataFrame(results)
pivot = (res_df.pivot(index='pollutant', columns='year', values='rows')
               .fillna(0).astype(int))
print('Rows per pollutant × year:')
print(pivot.to_string())

print('\nFailures / empties:')
bad = res_df[(res_df['rows'] == 0) | (res_df['http'] != 200)]
if len(bad):
    print(bad.to_string(index=False))
else:
    print('  none — all 50 requests succeeded with non-empty data.')

Rows per pollutant × year:
year        2016   2017   2018   2019   2020   2021   2022   2023   2024   2025
pollutant                                                                      
co         10119   9801   9878   9692   9051   8665   7066   6335   5760   5812
no2        10154   9704   9772  10562  11552  11376  10718  10206  10100  10130
ozone      18911  18298  18886  18756  19920  19758  18378  17360  17301  17375
pm25       25744  26667  26017  27165  26884  26655  20786  18442  18338  17841
so2         6543   6471   6512   6430   4180   5900   4334   4345   4342   4304

Failures / empties:
  none — all 50 requests succeeded with non-empty data.


## Merging api_pulls/ into one long table (5 pollutants)

Now we redo the daily-AQI computation using `api_pulls/` as the primary source, with all five pollutants (Ozone, PM2.5, CO, SO2, NO2). Two adjustments vs. the earlier 2-pollutant merge: (1) the AQS API returns one row per monitor-day **per NAAQS standard** (ozone has 4, PM2.5 has 6+, etc. — same underlying concentration, different breakpoints), so we dedupe by taking the max AQI per (site, date, pollutant); (2) about 18% of rows have null AQI (secondary standards for CO/SO2/NO2 don't generate one), so we drop those. The same `observation_percent >= 75` completeness threshold applies. Result is saved to `la_daily_aqi_5pollutants_2016_2025.csv` — the old 2-pollutant CSV is left in place for comparison.

In [19]:
POLLUTANT_LABEL = {'ozone':'Ozone','pm25':'PM2.5','co':'CO','so2':'SO2','no2':'NO2'}

# Load the 50 first-pass API files, tagging pollutant from the filename prefix.
#
# The pm25_88502_* files are explicitly EXCLUDED here. They are pulled further down
# the notebook, and this cell has to keep reproducing the *pre-fix* v1 series so the
# discrepancy investigation below still has something to investigate. Without this
# filter the notebook is not re-runnable: on a second pass the 88502 files are already
# on disk, v1 silently comes out identical to v2, and the diagnosis cells fail on an
# empty frame.
api_frames = []
for f in sorted((API_DIR).glob('*.csv')):
    if f.name.startswith('pm25_88502_'):
        continue
    df = pd.read_csv(f, low_memory=False)
    df['pollutant_type'] = POLLUTANT_LABEL[f.name.split('_')[0]]
    api_frames.append(df)
print(f'Loaded {len(api_frames)} first-pass files (88502 held back for the v2 rebuild)')
raw = pd.concat(api_frames, ignore_index=True)
print(f'Raw rows across all 50 files: {len(raw):,}')

# Drop rows with no AQI (secondary standards for CO/SO2/NO2 don't produce one)
raw = raw.dropna(subset=['aqi'])
# Apply completeness threshold
before = len(raw)
raw = raw[raw['observation_percent'] >= 75]
print(f'After aqi-not-null + observation_percent >= 75: {len(raw):,} '
      f'(dropped {before-len(raw):,})')

# Sanity: every row is California / Los Angeles (fail loudly if not)
assert (raw['state']  == 'California').all(),  'Non-CA rows found in api_pulls'
assert (raw['county'] == 'Los Angeles').all(), 'Non-LA rows found in api_pulls'

# Parse date + build a site_id from state+county+site codes
raw['date']    = pd.to_datetime(raw['date_local'], errors='coerce')
raw['site_id'] = (raw['state_code'].astype(str).str.zfill(2)
                + raw['county_code'].astype(str).str.zfill(3)
                + raw['site_number'].astype(str).str.zfill(4))

# Dedupe multiple standards per monitor-day: take max AQI across standards.
# (Ozone has 4 NAAQS standards, PM2.5 has 6+, etc. — same underlying reading,
# different breakpoints. Max = worst-case AQI for that monitor-day-pollutant.)
dedup = (raw.groupby(['date','site_id','local_site_name','pollutant_type'],
                     as_index=False)['aqi'].max())
print(f'After dedup to (site, date, pollutant): {len(dedup):,}')
print()
print('Rows per pollutant after dedup:')
print(dedup['pollutant_type'].value_counts().to_string())

Loaded 50 first-pass files (88502 held back for the v2 rebuild)
Raw rows across all 50 files: 659,296


After aqi-not-null + observation_percent >= 75: 528,041 (dropped 8,280)


After dedup to (site, date, pollutant): 163,437

Rows per pollutant after dedup:
pollutant_type
NO2      49197
Ozone    45138
CO       37424
PM2.5    22894
SO2       8784


In [20]:
daily_5pol = (dedup.groupby('date', as_index=False)
                    .agg(daily_aqi=('aqi','max'),
                         n_monitor_readings=('aqi','count'),
                         n_sites=('site_id','nunique'),
                         pollutants_used=('pollutant_type', lambda s: ','.join(sorted(s.unique())))))
idx_max = dedup.groupby('date')['aqi'].idxmax()
dom = dedup.loc[idx_max, ['date','pollutant_type','local_site_name']].rename(
    columns={'pollutant_type':'dominant_pollutant','local_site_name':'dominant_site'})
daily_5pol = (daily_5pol.merge(dom, on='date', how='left')
                        .sort_values('date').reset_index(drop=True))

# Coverage
full = pd.date_range('2016-01-01','2025-12-31', freq='D')
missing = full.difference(pd.DatetimeIndex(daily_5pol['date']))
print(f'Daily rows: {len(daily_5pol):,}')
print(f'Expected days: {len(full):,}   Covered: {len(daily_5pol):,}   Missing: {len(missing):,}')

# Distribution
print('\nDistribution:')
print(daily_5pol['daily_aqi'].describe(percentiles=[.5,.75,.9,.95,.99]).to_string())

# Dominant-pollutant breakdown — reveals which pollutants actually drive LA's daily AQI
print('\nDominant pollutant on daily-max day (across 3,653 days):')
print(daily_5pol['dominant_pollutant'].value_counts().to_string())

# Annual summary
print('\nAnnual summary:')
annual2 = (daily_5pol.assign(year=daily_5pol['date'].dt.year)
                     .groupby('year')
                     .agg(days=('daily_aqi','size'),
                          mean_aqi=('daily_aqi','mean'),
                          median_aqi=('daily_aqi','median'),
                          max_aqi=('daily_aqi','max'),
                          days_over_100=('daily_aqi', lambda s: (s>100).sum()),
                          days_over_150=('daily_aqi', lambda s: (s>150).sum())))
annual2['mean_aqi']   = annual2['mean_aqi'].round(1)
annual2['median_aqi'] = annual2['median_aqi'].round(1)
print(annual2.to_string())

# Save
OUT_CSV_5POL = PROCESSED / 'la_daily_aqi_5pollutants_2016_2025.csv'
daily_5pol.to_csv(OUT_CSV_5POL, index=False)
print(f'\nSaved: {OUT_CSV_5POL.name}  ({len(daily_5pol):,} rows, {OUT_CSV_5POL.stat().st_size/1024:.1f} KB)')

Daily rows: 3,653
Expected days: 3,653   Covered: 3,653   Missing: 0

Distribution:
count    3653.000000
mean       86.590200
std        39.149017
min        25.000000
50%        74.000000
75%       108.000000
90%       150.000000
95%       166.000000
99%       201.000000
max       250.000000

Dominant pollutant on daily-max day (across 3,653 days):
dominant_pollutant
Ozone    1942
PM2.5    1503
NO2       208

Annual summary:
      days  mean_aqi  median_aqi  max_aqi  days_over_100  days_over_150
year                                                                   
2016   366      84.9        74.0    210.0             98             23
2017   365      90.7        77.0    224.0            118             46
2018   365      84.7        76.0    201.0            106             18
2019   365      82.6        71.0    201.0             82             28
2020   366      97.5        84.5    250.0            134             58
2021   365      86.6        77.0    195.0             97          

### A note on re-running this notebook

Two idempotency issues were found and fixed on 2026-08-12, when the notebook was first
re-run end to end after the original session.

**The v1 build was picking up the fix it predates.** The cell that rebuilds the
5-pollutant v1 series globs everything in `api_pulls/`, and `pm25_88502_2019_api.csv`
splits on `_` to `pm25`. So on any second run — once the 88502 files exist on disk — v1
came out already containing the very data whose absence it is supposed to demonstrate,
the discrepancy investigation found zero days, and the cell below crashed on an empty
frame. That cell now skips `pm25_88502_*` explicitly.

**The committed v1 artifact was contaminated by the same bug.** All 119 affected days
trace to one 88502 monitor at Los Angeles-North Main Street. The true pre-fix v1 is
lower than the 2-pollutant baseline on **1,106** days; the previously committed artifact
showed 1,071 because it had partially absorbed the fix. The figures in this notebook and
in the README now read 1,106.

**None of this touches v2.** The final dataset,
`la_daily_aqi_5pollutants_v2_2016_2025.csv`, regenerates byte-identically, as does the
2-pollutant baseline. Only the kept-for-diff v1 artifact changed.

## Investigating why 5-pol < 2-pol on 1,106 days

Adding more pollutants to a max-based city-wide AQI should only push the number **up or unchanged** — never down. But the 5-pollutant series (from the AQS API) is *lower* than the 2-pollutant series (from AirData) on 1,106 out of 3,653 days. Something is wrong.

**Working hypothesis**: the AQS API returns multiple rows per site-day-pollutant, one per NAAQS "Pollutant Standard" (e.g. ozone has both a 1-hour 1979 standard and an 8-hour 2015 standard). If different standards produce different AQI values from the same measurement, the dedup step (max AQI across standards) could pick a "wrong" standard on some days and produce inconsistent results.

Below: identify the 1,106 days, sample 5 of them, and inspect the raw pre-dedup API rows. No files are modified — this is investigation only.

In [21]:
# Step 1: Identify the 1,106 dates where 5-pol < 2-pol; pick 5 sample dates.

old = pd.read_csv(PROCESSED / 'la_daily_aqi_2016_2025.csv')
new = pd.read_csv(PROCESSED / 'la_daily_aqi_5pollutants_2016_2025.csv')
old['date'] = pd.to_datetime(old['date'])
new['date'] = pd.to_datetime(new['date'])

merged = old[['date','daily_aqi','dominant_pollutant']].merge(
    new[['date','daily_aqi','dominant_pollutant']],
    on='date', suffixes=('_2pol','_5pol'))
merged['diff'] = merged['daily_aqi_5pol'] - merged['daily_aqi_2pol']
lower = merged[merged['diff'] < 0].sort_values('diff')

print(f'Days where 5-pol < 2-pol: {len(lower)}')
print(f'Diff range: min={lower["diff"].min()}, median={lower["diff"].median()}, max={lower["diff"].max()}')
print()

# Pick 5 samples spanning the diff range
samples = pd.concat([lower.head(2), lower.iloc[[len(lower)//2]], lower.tail(2)])
print('Five sample dates spanning worst → smallest diffs:')
print(samples.to_string(index=False))

Days where 5-pol < 2-pol: 1106
Diff range: min=-134.0, median=-6.0, max=-1.0

Five sample dates spanning worst → smallest diffs:
      date  daily_aqi_2pol dominant_pollutant_2pol  daily_aqi_5pol dominant_pollutant_5pol   diff
2019-10-11             197                   PM2.5            63.0                     NO2 -134.0
2020-09-11             223                   PM2.5           134.0                   PM2.5  -89.0
2023-10-26              54                   PM2.5            48.0                   PM2.5   -6.0
2019-07-18              62                   PM2.5            61.0                   Ozone   -1.0
2019-12-29              84                   PM2.5            83.0                   PM2.5   -1.0


In [22]:
# Step 2: Pull RAW (pre-dedup) rows for one high-diff day; show all columns
# related to standard, sample duration, AQI. Focus on 2019-10-11 (the -134 day).

d = '2019-10-11'
pm_cols = ['local_site_name','poc','sample_duration','pollutant_standard',
           'first_max_value','arithmetic_mean','aqi','observation_percent']
oz_cols = pm_cols  # same schema

pm_raw = pd.read_csv(API_DIR/'pm25_2019_api.csv', low_memory=False)
oz_raw = pd.read_csv(API_DIR/'ozone_2019_api.csv', low_memory=False)

print(f'=== RAW API PM2.5 rows for {d} — {len(pm_raw[pm_raw["date_local"]==d])} rows ===')
print(pm_raw[pm_raw['date_local']==d][pm_cols].to_string(index=False))
print(f'\n=== RAW API Ozone rows for {d} — {len(oz_raw[oz_raw["date_local"]==d])} rows ===')
print(oz_raw[oz_raw['date_local']==d][oz_cols].to_string(index=False))

=== RAW API PM2.5 rows for 2019-10-11 — 51 rows ===
               local_site_name  poc sample_duration pollutant_standard  first_max_value  arithmetic_mean  aqi  observation_percent
     Lancaster-Division Street    1          1 HOUR                NaN              9.0         3.958333  NaN                100.0
            Long Beach (South)    3          1 HOUR                NaN             33.3        13.066667  NaN                100.0
Long Beach-Route 710 Near Road    3          1 HOUR                NaN             60.9        51.200000  NaN                  8.0
                       Compton    1         24 HOUR  PM25 24-hour 2006             11.5        11.500000 55.0                100.0
                       Compton    1         24 HOUR   PM25 Annual 2006             11.5        11.500000 55.0                100.0
                       Compton    1         24 HOUR  PM25 24-hour 2012             11.5        11.500000 55.0                100.0
                       Compton 

**Observation from the raw rows above.** For a given monitor on a given day, all pollutant_standard values produce the **same** AQI (Long Beach POC=1 has AQI=58 across 8 different PM2.5 standards; Reseda ozone POC=1 has AQI=61 across 3 different 8-hour standards). So the standards-multiplicity hypothesis doesn't cause the discrepancy. Also, the maximum PM2.5 AQI in the entire API dataset for 2019-10-11 is only 58 — but the 2-pollutant dataset says PM2.5 hit 197. There must be a monitor in AirData that's missing from the API pull. Let's look at AirData for the same day.

In [23]:
# Step 3: Real cause — check AirData for the missing high-AQI monitor, and quantify scope.

d_air = '10/11/2019'  # AirData date format
print(f'=== AirData la_only/pm25_2019.csv rows for {d_air}, sorted by AQI desc ===')
pm_air = pd.read_csv(LA_ONLY/'pm25_2019.csv', low_memory=False)
r_air = pm_air[pm_air['Date'] == d_air][
    ['Local Site Name','POC','AQS Parameter Code','AQS Parameter Description',
     'Daily Mean PM2.5 Concentration','Daily AQI Value']]
print(r_air.sort_values('Daily AQI Value', ascending=False).to_string(index=False))

print('\n=== Parameter codes in the AirData PM2.5 dataset across all 10 years ===')
pm_all = pd.concat([
    pd.read_csv(f, low_memory=False,
                usecols=['AQS Parameter Code','AQS Parameter Description'])
    for f in sorted((LA_ONLY).glob('pm25_*.csv'))
])
print(pm_all.groupby(['AQS Parameter Code','AQS Parameter Description']).size().to_string())
c = pm_all['AQS Parameter Code'].value_counts()
print(f'\n88502 share of AirData PM2.5 rows: {100*c.get(88502,0)/c.sum():.1f}%')

print('\n=== Confirm on second high-diff day: 2020-09-11 ===')
pm20 = pd.read_csv(LA_ONLY/'pm25_2020.csv', low_memory=False)
r20 = pm20[pm20['Date']=='09/11/2020'][
    ['Local Site Name','POC','AQS Parameter Code',
     'Daily Mean PM2.5 Concentration','Daily AQI Value']]
print('AirData top 5 by AQI (2020-09-11):')
print(r20.sort_values('Daily AQI Value', ascending=False).head(5).to_string(index=False))
print('\nAPI top 5 by AQI per site (2020-09-11):')
api20 = pd.read_csv(API_DIR/'pm25_2020_api.csv', low_memory=False)
top = (api20[api20['date_local']=='2020-09-11']
       .dropna(subset=['aqi'])
       .groupby(['local_site_name','poc'])['aqi'].max()
       .sort_values(ascending=False).head(5))
print(top.to_string())

=== AirData la_only/pm25_2019.csv rows for 10/11/2019, sorted by AQI desc ===
               Local Site Name  POC  AQS Parameter Code              AQS Parameter Description  Daily Mean PM2.5 Concentration  Daily AQI Value
                        Reseda    3               88502 Acceptable PM2.5 AQI & Speciation Mass                           120.9              197
            Long Beach (South)    3               88101               PM2.5 - Local Conditions                            13.0               58
Long Beach-Route 710 Near Road    1               88101               PM2.5 - Local Conditions                            13.0               58
 Los Angeles-North Main Street    9               88502 Acceptable PM2.5 AQI & Speciation Mass                            11.6               56
                       Compton    1               88101               PM2.5 - Local Conditions                            11.5               55
 Los Angeles-North Main Street    1               88101   

AQS Parameter Code  AQS Parameter Description             
88101               PM2.5 - Local Conditions                  30292
88502               Acceptable PM2.5 AQI & Speciation Mass    19217

88502 share of AirData PM2.5 rows: 38.8%

=== Confirm on second high-diff day: 2020-09-11 ===
AirData top 5 by AQI (2020-09-11):
               Local Site Name  POC  AQS Parameter Code  Daily Mean PM2.5 Concentration  Daily AQI Value
                      Glendora    3               88502                           148.1              223
 Los Angeles-North Main Street    9               88502                            52.6              143
 Los Angeles-North Main Street    1               88101                            48.7              134
Long Beach-Route 710 Near Road    3               88101                            44.3              123
                       Compton    3               88502                            43.0              119

API top 5 by AQI per site (2020-09-11):


local_site_name                 poc
Los Angeles-North Main Street   1      134.0
Long Beach-Route 710 Near Road  3      123.0
Compton                         1      114.0
Long Beach-Route 710 Near Road  1      109.0
Long Beach (South)              3      106.0


## Findings

**Hypothesis (multiple pollutant standards per site-day producing different AQIs): rejected.**
The API does return multiple rows per site-day-pollutant (ozone has 4 standards, PM2.5 has 6+), but **every standard produces the same AQI for the same monitor-day**. Taking the max across standards is a no-op — it never changes the value. So the dedup step is not the source of the discrepancy.

**Actual cause: the API pull only fetched param code 88101 (PM2.5 - Local Conditions), but AirData bundles two PM2.5 parameters.**
The AirData PM2.5 exports contain **both**:
- `88101` PM2.5 - Local Conditions — 30,292 rows (61.2%)
- `88502` Acceptable PM2.5 AQI & Speciation Mass — 19,217 rows (38.8%)

Param 88502 is a separate EPA measurement (from the chemical speciation network) that EPA also treats as valid AQI-quality PM2.5 data. On many days, 88502 monitors report *higher* PM2.5 than 88101 monitors — the extreme 2019-10-11 case had Reseda POC=3 at 120.9 ug/m3 → AQI 197 (all under 88502), while the highest 88101 reading anywhere in LA that day was 13.0 ug/m3 → AQI 58.

That 38.8% chunk of PM2.5 data is entirely absent from the 5-pollutant dataset, which explains every one of the 1,106 days where 5-pol < 2-pol.

**On other pollutants**: the AirData ozone exports contain only param 44201 (matches the API), so ozone is not affected. CO/SO2/NO2 were never in AirData, so there's no old-vs-new comparison for those — but it's worth checking whether any of them also have secondary "acceptable AQI" parameter codes before finalizing.

**Suggested fix** (pending your decision — I'm not touching anything until you say):
Add a second API pull for `param=88502` × 10 years, merge it in alongside the 88101 data, re-dedupe by (site, date, pollutant) with max AQI. Should take ~1 minute (10 requests, 6s apart) and should close the entire 1,106-day gap.

Also worth considering: query EPA's parameter class list to see if CO/SO2/NO2 have analogous "acceptable AQI" codes we should pull in for completeness.

## Applying the fix — pull 88502 and rebuild as v2

Two steps before we rebuild:
1. **Confirm what parameter codes actually produce AQI values.** Before assuming CO/SO2/NO2 might have secondary codes analogous to PM2.5's 88502, we ask EPA's `list/parametersByClass?pc=AQI+POLLUTANTS` endpoint for the authoritative list.
2. **Pull param 88502 for 2016–2025** — the missing "Acceptable PM2.5 AQI & Speciation Mass" data. Same rate limits and CA/LA assertions as the original pull.

Then we rebuild the daily-AQI dataset and save it as `la_daily_aqi_5pollutants_v2_2016_2025.csv` (v1 stays untouched so we can diff the two).

In [24]:
# Verify EPA's parameter class list before pulling anything else.
# We want to confirm which parameter codes actually produce AQI values —
# specifically, whether CO/SO2/NO2 have secondary "acceptable AQI" codes like 88502 does for PM2.5.

for cls in ('AQI POLLUTANTS', 'AIRNOW MAPS'):
    r = requests.get('https://aqs.epa.gov/data/api/list/parametersByClass',
                     params={'email': AQS_EMAIL, 'key': AQS_KEY, 'pc': cls}, timeout=30)
    print(f'=== {cls} ===')
    for row in r.json().get('Data', []):
        print(f"  {row['code']:>6s}  {row['value_represented']}")
    print()

=== AQI POLLUTANTS ===
   42101  Carbon monoxide
   42401  Sulfur dioxide
   42602  Nitrogen dioxide (NO2)
   44201  Ozone
   81102  PM10 Total 0-10um STP
   88101  PM2.5 - Local Conditions
   88502  Acceptable PM2.5 AQI & Speciation Mass



=== AIRNOW MAPS ===
   44201  Ozone
   88101  PM2.5 - Local Conditions
   88502  Acceptable PM2.5 AQI & Speciation Mass



In [25]:
# Pull param 88502 (Acceptable PM2.5 AQI & Speciation Mass) for 2016–2025.
# Idempotent: skips years already saved to api_pulls/.

API_URL = 'https://aqs.epa.gov/data/api/dailyData/byCounty'
STATE, COUNTY = '06', '037'
DELAY_SEC = 6.0
API_DIR = API_DIR

pull_88502 = []
for year in range(2016, 2026):
    out = API_DIR / f'pm25_88502_{year}_api.csv'
    if out.exists():
        n_existing = len(pd.read_csv(out, low_memory=False))
        print(f'  88502 {year}  SKIP (already have {n_existing:,} rows in {out.name})')
        pull_88502.append({'year': year, 'rows': n_existing, 'note': 'already-cached'})
        continue
    r = requests.get(API_URL, params={
        'email': AQS_EMAIL, 'key': AQS_KEY, 'param': '88502',
        'bdate': f'{year}0101', 'edate': f'{year}1231',
        'state': STATE, 'county': COUNTY,
    }, timeout=60)
    payload = r.json(); data = payload.get('Data', []) or []
    df_api = pd.DataFrame(data)
    if len(df_api):
        assert (df_api['state']  == 'California').all()
        assert (df_api['county'] == 'Los Angeles').all()
        df_api.to_csv(out, index=False)
    print(f'  88502 {year}  HTTP {r.status_code}  rows={len(df_api):<5d}  saved {out.name}')
    pull_88502.append({'year': year, 'rows': len(df_api), 'note': 'fetched'})
    time.sleep(DELAY_SEC)

print(f'\n88502 pull complete. Row counts:')
for r in pull_88502: print(f'  {r["year"]}: {r["rows"]:,}')

  88502 2016  SKIP (already have 3,084 rows in pm25_88502_2016_api.csv)
  88502 2017  SKIP (already have 2,998 rows in pm25_88502_2017_api.csv)
  88502 2018  SKIP (already have 3,075 rows in pm25_88502_2018_api.csv)
  88502 2019  SKIP (already have 3,106 rows in pm25_88502_2019_api.csv)
  88502 2020  SKIP (already have 4,153 rows in pm25_88502_2020_api.csv)
  88502 2021  SKIP (already have 4,455 rows in pm25_88502_2021_api.csv)
  88502 2022  SKIP (already have 4,974 rows in pm25_88502_2022_api.csv)
  88502 2023  SKIP (already have 5,121 rows in pm25_88502_2023_api.csv)
  88502 2024  SKIP (already have 5,085 rows in pm25_88502_2024_api.csv)
  88502 2025  SKIP (already have 5,116 rows in pm25_88502_2025_api.csv)

88502 pull complete. Row counts:
  2016: 3,084
  2017: 2,998
  2018: 3,075
  2019: 3,106
  2020: 4,153
  2021: 4,455
  2022: 4,974
  2023: 5,121
  2024: 5,085
  2025: 5,116


In [26]:
# Rebuild the daily AQI with 88502 included. Saved as v2, keeping v1 intact for diff-checking.
POL_LABEL = {'ozone':'Ozone','pm25':'PM2.5','co':'CO','so2':'SO2','no2':'NO2'}

frames = []
for f in sorted((API_DIR).glob('*.csv')):
    df = pd.read_csv(f, low_memory=False)
    # pm25_88502_YYYY_api.csv and pm25_YYYY_api.csv both map to 'PM2.5'
    if f.name.startswith('pm25_88502_'):
        df['pollutant_type'] = 'PM2.5'
    else:
        df['pollutant_type'] = POL_LABEL[f.name.split('_')[0]]
    frames.append(df)
raw = pd.concat(frames, ignore_index=True)
print(f'Total raw rows (all API files including 88502): {len(raw):,}')
print('\nRow counts per parameter_code:')
print(raw['parameter_code'].value_counts().to_string())

# Same filtering as v1
raw = raw.dropna(subset=['aqi'])
before = len(raw)
raw = raw[raw['observation_percent'] >= 75]
print(f'\nAfter aqi-not-null + observation_percent >= 75: {len(raw):,} (dropped {before-len(raw):,})')

assert (raw['state']  == 'California').all()
assert (raw['county'] == 'Los Angeles').all()

raw['date']    = pd.to_datetime(raw['date_local'], errors='coerce')
raw['site_id'] = (raw['state_code'].astype(str).str.zfill(2)
                + raw['county_code'].astype(str).str.zfill(3)
                + raw['site_number'].astype(str).str.zfill(4))

# Dedup by (site+poc, date, pollutant) — treats co-located monitors as distinct,
# collapses multi-standard rows via max AQI (same value anyway).
dedup = (raw.groupby(['date','site_id','poc','local_site_name','pollutant_type'],
                     as_index=False)['aqi'].max())
print(f'\nAfter dedup (site+poc, date, pollutant): {len(dedup):,}')
print('\nRows per pollutant after dedup:')
print(dedup['pollutant_type'].value_counts().to_string())

# City-wide daily AQI = max across all monitors and pollutants
daily_v2 = (dedup.groupby('date', as_index=False)
                  .agg(daily_aqi=('aqi','max'),
                       n_monitor_readings=('aqi','count'),
                       n_sites=('site_id','nunique'),
                       pollutants_used=('pollutant_type', lambda s: ','.join(sorted(s.unique())))))
idx = dedup.groupby('date')['aqi'].idxmax()
dom = dedup.loc[idx, ['date','pollutant_type','local_site_name']].rename(
    columns={'pollutant_type':'dominant_pollutant','local_site_name':'dominant_site'})
daily_v2 = (daily_v2.merge(dom, on='date', how='left')
                    .sort_values('date').reset_index(drop=True))

OUT_V2 = PROCESSED / 'la_daily_aqi_5pollutants_v2_2016_2025.csv'
daily_v2.to_csv(OUT_V2, index=False)
print(f'\nSaved: {OUT_V2.name}  ({len(daily_v2):,} rows, {OUT_V2.stat().st_size/1024:.1f} KB)')

Total raw rows (all API files including 88502): 700,463

Row counts per parameter_code:
parameter_code
88101    234539
44201    184943
42602    104274
42101     82179
42401     53361
88502     41167



After aqi-not-null + observation_percent >= 75: 549,394 (dropped 8,280)



After dedup (site+poc, date, pollutant): 194,341

Rows per pollutant after dedup:
pollutant_type
NO2      51503
PM2.5    49494
Ozone    45138
CO       39422
SO2       8784

Saved: la_daily_aqi_5pollutants_v2_2016_2025.csv  (3,653 rows, 251.3 KB)


In [27]:
# Compare v2 vs 2-pol dataset — did the fix close the 1,106-day gap?
old = pd.read_csv(PROCESSED / 'la_daily_aqi_2016_2025.csv');            old['date'] = pd.to_datetime(old['date'])
v1  = pd.read_csv(PROCESSED / 'la_daily_aqi_5pollutants_2016_2025.csv'); v1['date'] = pd.to_datetime(v1['date'])
v2  = pd.read_csv(OUT_V2);                                       v2['date'] = pd.to_datetime(v2['date'])

m = old[['date','daily_aqi']].merge(v2[['date','daily_aqi']], on='date', suffixes=('_2pol','_v2'))
m['diff'] = m['daily_aqi_v2'] - m['daily_aqi_2pol']
print('v2 vs 2-pol:')
print(f'  v2 < 2-pol : {(m["diff"]<0).sum():>5}   (was 1,106 in v1)')
print(f'  v2 = 2-pol : {(m["diff"]==0).sum():>5}   (was 2,450 in v1)')
print(f'  v2 > 2-pol : {(m["diff"]>0).sum():>5}   (was 132 in v1)')
print(f'  mean(v2)   : {m["daily_aqi_v2"].mean():.2f}   mean(2-pol): {m["daily_aqi_2pol"].mean():.2f}')

# Monotonicity check: v2 must never be less than v1 (only added data, never removed)
m12 = v1[['date','daily_aqi']].merge(v2[['date','daily_aqi']], on='date', suffixes=('_v1','_v2'))
print('\nv2 vs v1 monotonicity:')
print(f'  v2 > v1 : {(m12["daily_aqi_v2"]>m12["daily_aqi_v1"]).sum()}')
print(f'  v2 = v1 : {(m12["daily_aqi_v2"]==m12["daily_aqi_v1"]).sum()}')
print(f'  v2 < v1 : {(m12["daily_aqi_v2"]<m12["daily_aqi_v1"]).sum()}   (must be 0)')

# Updated distribution + dominant pollutant + annual summary
print('\n=== Distribution (v2) ===')
print(v2['daily_aqi'].describe(percentiles=[.5,.75,.9,.95,.99]).to_string())

print('\n=== Dominant pollutant on daily-max day (v2) ===')
print(v2['dominant_pollutant'].value_counts().to_string())

print('\n=== Annual summary (v2) ===')
annual2 = (v2.assign(year=v2['date'].dt.year)
             .groupby('year')
             .agg(days=('daily_aqi','size'),
                  mean_aqi=('daily_aqi','mean'),
                  median_aqi=('daily_aqi','median'),
                  max_aqi=('daily_aqi','max'),
                  days_over_100=('daily_aqi', lambda s: (s>100).sum()),
                  days_over_150=('daily_aqi', lambda s: (s>150).sum())))
annual2['mean_aqi']=annual2['mean_aqi'].round(1)
annual2['median_aqi']=annual2['median_aqi'].round(1)
print(annual2.to_string())

v2 vs 2-pol:
  v2 < 2-pol :     0   (was 1,106 in v1)
  v2 = 2-pol :  3521   (was 2,450 in v1)
  v2 > 2-pol :   132   (was 132 in v1)
  mean(v2)   : 88.75   mean(2-pol): 88.47

v2 vs v1 monotonicity:
  v2 > v1 : 1106
  v2 = v1 : 2547
  v2 < v1 : 0   (must be 0)

=== Distribution (v2) ===
count    3653.000000
mean       88.751163
std        38.047805
min        31.000000
50%        77.000000
75%       108.000000
90%       150.000000
95%       169.000000
99%       202.480000
max       250.000000

=== Dominant pollutant on daily-max day (v2) ===
dominant_pollutant
PM2.5    1856
Ozone    1663
NO2       134

=== Annual summary (v2) ===
      days  mean_aqi  median_aqi  max_aqi  days_over_100  days_over_150
year                                                                   
2016   366      88.1        77.0    226.0            104             23
2017   365      93.5        79.0    224.0            119             46
2018   365      87.6        78.0    201.0            108             19
2